In [1]:
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

print("=== 06a: Contextual Bit Repair — Environment Check ===")
print("Python:", sys.version.split()[0])
print("Drive accessible:", Path("/content/drive/MyDrive").is_dir())

for package in ["torch", "numpy", "ultralytics", "scikit-learn"]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch unavailable.")

print("No packages installed. No files modified. No inference performed.")

=== 06a: Contextual Bit Repair — Environment Check ===
Python: 3.13.18
Drive accessible: False
torch: 2.14.1+cpu
numpy: 2.4.6
ultralytics: NOT INSTALLED
scikit-learn: 1.6.1
CUDA available: False
No packages installed. No files modified. No inference performed.


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
from pathlib import Path
import hashlib
import json
import subprocess

root = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
repo = root / "code"

head = subprocess.run(
    ["git", "-C", str(repo), "rev-parse", "HEAD"],
    capture_output=True, text=True, check=True,
).stdout.strip()

if head != "710335d2f526899ab1b27c1f34c80660b2a3b199":
    raise RuntimeError(f"Unexpected repository HEAD: {head}")

status = subprocess.run(
    ["git", "-C", str(repo), "status", "--short"],
    capture_output=True, text=True, check=True,
).stdout.strip()

sources = {
    "Checkpoint": (
        root / "models/yolov8s.pt",
        "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a",
    ),
    "Published baseline report": (
        repo / "docs/research_reports/05a_shared_INT8_clean_evaluation_v1.json",
        "47861220401cc36ea2c6c19f427d351dcbb5ed39a9067fa79e0abbb175f889d7",
    ),
}

for label, (path, expected_sha) in sources.items():
    actual_sha = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual_sha != expected_sha:
        raise RuntimeError(f"{label} SHA256 mismatch.")
    print(f"{label}: PASS")

print("Git HEAD:", head)
print("Working tree:", status or "(clean)")
print("No model loaded. No files modified. No inference performed.")

Checkpoint: PASS
Published baseline report: PASS
Git HEAD: 710335d2f526899ab1b27c1f34c80660b2a3b199
Working tree: (clean)
No model loaded. No files modified. No inference performed.


In [6]:
from pathlib import Path
import hashlib
import json

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
destination = (
    repo / "experiments/notebook06a/masked_bit_feasibility_protocol_v1.json"
)

protocol = {
    "status": "FROZEN_INITIAL_FEASIBILITY_PROTOCOL",
    "source_commit": "710335d2f526899ab1b27c1f34c80660b2a3b199",
    "seed": 20261116,
    "representation": "05a_shared_logical_INT8_per_output_channel",
    "scope": "63_eligible_Conv2d_weight_tensors",
    "task": "predict_one_masked_clean_bit_at_a_known_weight_and_bit_position",
    "bits": list(range(8)),
    "split": {
        "unit": "output_channel",
        "within_each_layer": "seeded_60_20_20_train_calibration_test",
        "channel_groups_disjoint": True,
        "all_context_weights_must_remain_in_the_target_output_channel": True,
        "maximum_target_weights_per_layer": {
            "train": 512,
            "calibration": 256,
            "test": 256,
        },
        "sampling": "uniform_without_replacement_within_each_partition",
    },
    "comparisons": {
        "A": "smoothed_bit_frequency_by_layer_and_bit",
        "B": "smoothed_conditional_frequency_given_other_7_bits_and_layer",
        "C": "other_7_bits_and_layer_plus_local_weight_context",
    },
    "context": {
        "definition": "up_to_8_previous_and_8_next_flattened_weights_in_same_output_channel",
        "target_weight_excluded": True,
        "boundary_handling": "available_neighbors_only_with_count_feature",
        "features": [
            "neighbor_integer_mean",
            "neighbor_integer_standard_deviation",
            "neighbor_mean_absolute_integer",
            "neighbor_zero_fraction",
            "neighbor_bit_frequencies_0_to_7",
            "neighbor_count",
        ],
        "assumption": "context_is_clean_in_this_initial_completion_probe",
    },
    "predictor_C": {
        "family": "HistGradientBoostingClassifier_one_model_per_bit",
        "max_iter": 100,
        "max_leaf_nodes": 15,
        "learning_rate": 0.1,
        "min_samples_leaf": 20,
        "l2_regularization": 1.0,
        "hyperparameter_search": False,
    },
    "leakage_controls": [
        "Mask the target bit before constructing target-weight features.",
        "Do not use the original target integer as a feature.",
        "Do not use weight address, flat index or output-channel identity.",
        "Do not use target-inclusive context statistics or FP32 target weights.",
        "Do not use test labels for fitting, calibration or method selection.",
    ],
    "evaluation": {
        "primary": "paired_test_log_loss_C_minus_B",
        "secondary": [
            "Brier_score",
            "accuracy_with_class_prevalence",
            "per_layer_and_per_bit_results",
        ],
        "probability_clipping": 1e-6,
        "frequency_smoothing": "Laplace_alpha_1",
        "uncertainty_unit": "output_channel_not_individual_weight",
        "constant_label_groups": "retain_and_report_metric_availability",
    },
    "limitations": [
        "Bit completion is not fault detection or successful repair.",
        "Known masked-bit position is not available from an unknown fault.",
        "Clean context does not establish robustness to corrupted neighbors.",
        "Predictive accuracy does not establish recovery of detection AP.",
        "Predictor memory, trusted state and runtime costs remain to be measured.",
        "No COCO images or held-out detection images are used in this probe.",
    ],
    "current_INT8_screening_campaign_changed": False,
}

content = (
    json.dumps(protocol, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

destination.parent.mkdir(parents=True, exist_ok=True)
if destination.exists():
    if destination.read_bytes() != content:
        raise RuntimeError("Existing protocol differs; do not overwrite.")
else:
    with destination.open("xb") as handle:
        handle.write(content)

if destination.read_bytes() != content:
    raise RuntimeError("Protocol write verification failed.")

print("Saved:", destination)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No model loaded. No predictor trained. No test results observed.")
print("Existing screening campaign unchanged. No commit or push.")

Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/experiments/notebook06a/masked_bit_feasibility_protocol_v1.json
SHA256: 4551de5d9702761eb29628ecbe57df3937785510ff09f609a5a8c072389a4447
No model loaded. No predictor trained. No test results observed.
Existing screening campaign unchanged. No commit or push.


In [7]:
import subprocess
import sys
from importlib.metadata import version

protected_packages = [
    "torch", "torchvision", "numpy", "scikit-learn",
]
before = {name: version(name) for name in protected_packages}

packages = [
    "ultralytics==8.4.189",
    "nvidia-ml-py==13.615.71",
    "ultralytics-thop==2.2.5",
    "ultralytics-platform==0.1.108",
    "jedi==0.20.1",
]

result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps", "--disable-pip-version-check",
        *packages,
    ],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Dependency installation failed.")

after = {name: version(name) for name in protected_packages}
if after != before:
    raise RuntimeError("A protected package version changed.")

check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
)
print("--- Dependency check ---")
print(check.stdout.strip())
print(check.stderr.strip())
if check.returncode != 0:
    raise RuntimeError("Dependency compatibility check failed.")

import torch
import ultralytics
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier

print("\nRequired imports: PASS")
for name in protected_packages:
    print(f"{name}: {version(name)}")
print("ultralytics:", ultralytics.__version__)
print("Protected package versions unchanged: PASS")
print("No model loaded. No predictor trained. No inference performed.")

--- Dependency check ---
No broken requirements found.

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.

Required imports: PASS
torch: 2.14.1+cpu
torchvision: 0.29.1+cpu
numpy: 2.4.6
scikit-learn: 1.6.1
ultralytics: 8.4.189
Protected package versions unchanged: PASS
No model loaded. No predictor trained. No inference performed.


In [8]:
from pathlib import Path
import hashlib
import json
import torch
from ultralytics import YOLO

root = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
repo = root / "code"

def sha256(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":"),
    ).encode("utf-8")

checkpoint = root / "models/yolov8s.pt"
if sha256(checkpoint.read_bytes()) != (
    "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a"
):
    raise RuntimeError("Checkpoint identity mismatch.")

protocol_path = (
    repo / "experiments/notebook06a/masked_bit_feasibility_protocol_v1.json"
)
if sha256(protocol_path.read_bytes()) != (
    "4551de5d9702761eb29628ecbe57df3937785510ff09f609a5a8c072389a4447"
):
    raise RuntimeError("Frozen feasibility protocol mismatch.")

report_bytes = (
    repo / "docs/research_reports/05a_shared_INT8_clean_evaluation_v1.json"
).read_bytes()
if sha256(report_bytes) != (
    "47861220401cc36ea2c6c19f427d351dcbb5ed39a9067fa79e0abbb175f889d7"
):
    raise RuntimeError("Baseline report identity mismatch.")

config05a = json.loads(report_bytes)["evaluation_record"]["payload"]["config"]
model06a = YOLO(str(checkpoint)).model.cpu().float().eval()

quantized_weights06a = {}
inventory06a = []

with torch.no_grad():
    for name, module in model06a.named_modules():
        if not isinstance(module, torch.nn.Conv2d):
            continue
        if name == "model.22.dfl.conv":
            continue

        maxima = module.weight.abs().amax(dim=(1, 2, 3), keepdim=True)
        scales = torch.where(
            maxima > 0, maxima / 127, torch.ones_like(maxima)
        )
        if not torch.isfinite(scales).all() or not (scales > 0).all():
            raise RuntimeError(f"Invalid scales: {name}")

        q = torch.round(
            module.weight / scales
        ).clamp(-127, 127).to(torch.int8)
        module.weight.copy_(q.float() * scales)

        quantized_weights06a[name] = q.cpu().numpy().copy()
        inventory06a.append({
            "layer": name,
            "shape": list(q.shape),
            "weights": q.numel(),
            "output_channels": q.shape[0],
        })

if (
    len(inventory06a) != 63
    or sum(row["weights"] for row in inventory06a) != 11146080
):
    raise RuntimeError("Quantized weight scope mismatch.")

with torch.inference_mode():
    model06a(torch.zeros(1, 3, 640, 640))

state_digest = hashlib.sha256()
for name, tensor in sorted(model06a.state_dict().items()):
    array = tensor.detach().cpu().contiguous().numpy()
    state_digest.update(canonical([name, str(array.dtype), list(array.shape)]))
    state_digest.update(array.tobytes())

if state_digest.hexdigest() != (
    config05a["model_state_sha256"]["int8_channel_clean"]
):
    raise RuntimeError("Reconstructed model differs from the 05a baseline.")

print("Shared INT8 model state matches 05a: PASS")
print("Eligible layers:", len(inventory06a))
print("Quantized weights:", sum(row["weights"] for row in inventory06a))
print("Stored arrays contain logical INT8 integers only.")
print("One synthetic forward completed; no COCO images used.")
print("No predictor trained. No files modified. Checkpoint unchanged.")

Shared INT8 model state matches 05a: PASS
Eligible layers: 63
Quantized weights: 11146080
Stored arrays contain logical INT8 integers only.
One synthetic forward completed; no COCO images used.
No predictor trained. No files modified. Checkpoint unchanged.


In [9]:
from pathlib import Path
import copy
import hashlib
import json

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
source = (
    repo / "experiments/notebook06a/masked_bit_feasibility_protocol_v1.json"
)
destination = (
    repo / "experiments/notebook06a/masked_bit_feasibility_protocol_v2.json"
)

source_bytes = source.read_bytes()
source_sha = hashlib.sha256(source_bytes).hexdigest()
if source_sha != (
    "4551de5d9702761eb29628ecbe57df3937785510ff09f609a5a8c072389a4447"
):
    raise RuntimeError("Initial protocol SHA256 mismatch.")

protocol06a = copy.deepcopy(json.loads(source_bytes))
protocol06a["version"] = "2.0"
protocol06a["previous_protocol_sha256"] = source_sha
protocol06a["amendment"] = {
    "timing": "after_weight_extraction_before_any_predictor_training_or_test_evaluation",
    "reason": "separate_neighbor_information_gain_from_predictor_family_effect",
    "previous_protocol_preserved": True,
}

protocol06a["comparisons"]["D"] = (
    "same_predictor_and_settings_as_C_using_other_7_bits_and_layer_only"
)
protocol06a["evaluation"]["secondary"].append(
    "paired_test_log_loss_C_minus_D_matched_predictor_control"
)
protocol06a["interpretation_rule"] = (
    "C beating B alone does not establish useful neighbor information; "
    "report C versus D with paired output-channel-level uncertainty."
)
protocol06a["predictor_D"] = copy.deepcopy(protocol06a["predictor_C"])

content = (
    json.dumps(protocol06a, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

if destination.exists():
    if destination.read_bytes() != content:
        raise RuntimeError("Existing v2 protocol differs; do not overwrite.")
else:
    with destination.open("xb") as handle:
        handle.write(content)

if destination.read_bytes() != content:
    raise RuntimeError("Protocol verification failed.")

print("Saved:", destination)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("Comparisons:", sorted(protocol06a["comparisons"]))
print("Original protocol preserved.")
print("No predictor trained. No test results observed. No commit or push.")

Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/experiments/notebook06a/masked_bit_feasibility_protocol_v2.json
SHA256: d851b7a91569ffa155d3e85ffd7afa1fb80e0df851857979cde7a51e392bceec
Comparisons: ['A', 'B', 'C', 'D']
Original protocol preserved.
No predictor trained. No test results observed. No commit or push.


In [10]:
from pathlib import Path
import hashlib
import io
import json
import numpy as np

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
protocol_path = (
    repo / "experiments/notebook06a/masked_bit_feasibility_protocol_v2.json"
)
protocol_bytes = protocol_path.read_bytes()
if hashlib.sha256(protocol_bytes).hexdigest() != (
    "d851b7a91569ffa155d3e85ffd7afa1fb80e0df851857979cde7a51e392bceec"
):
    raise RuntimeError("Protocol v2 SHA256 mismatch.")

protocol06a = json.loads(protocol_bytes)
output_dir = repo / "results/notebook06a/masked_bit_dataset_v1"
if output_dir.exists():
    raise FileExistsError("Dataset destination exists; inspect before rerunning.")

partitions = ("train", "calibration", "test")
collected = {
    part: {key: [] for key in ("bits", "context", "layer", "channel", "flat_index")}
    for part in partitions
}
split_inventory = []
offsets = np.array([*range(-8, 0), *range(1, 9)])

def neighbor_features(q, flat_indices):
    matrix = q.reshape(q.shape[0], -1).astype(np.int16)
    channel = flat_indices // matrix.shape[1]
    position = flat_indices % matrix.shape[1]
    neighbor_positions = position[:, None] + offsets
    valid = (
        (neighbor_positions >= 0)
        & (neighbor_positions < matrix.shape[1])
    )
    neighbors = matrix[
        channel[:, None],
        np.clip(neighbor_positions, 0, matrix.shape[1] - 1),
    ]
    count = valid.sum(axis=1)
    if np.any(count == 0):
        raise RuntimeError("Target has no available context.")
    values = np.where(valid, neighbors, 0).astype(np.float64)
    mean = values.sum(axis=1) / count
    variance = (values ** 2).sum(axis=1) / count - mean ** 2
    bit_frequencies = np.column_stack([
        ((((neighbors & 255) >> bit) & 1) * valid).sum(axis=1) / count
        for bit in range(8)
    ])
    return np.column_stack([
        mean,
        np.sqrt(np.maximum(variance, 0)),
        np.abs(values).sum(axis=1) / count,
        ((neighbors == 0) & valid).sum(axis=1) / count,
        bit_frequencies,
        count,
    ]).astype(np.float32)

for layer_id, row in enumerate(inventory06a):
    name = row["layer"]
    q = quantized_weights06a[name]
    if list(q.shape) != row["shape"] or q.dtype != np.int8:
        raise RuntimeError(f"Integer array mismatch: {name}")

    seed_bytes = json.dumps(
        [protocol06a["seed"], name], separators=(",", ":")
    ).encode("utf-8")
    layer_seed = int(hashlib.sha256(seed_bytes).hexdigest()[:16], 16)
    rng = np.random.Generator(np.random.PCG64(layer_seed))

    channel_order = rng.permutation(q.shape[0])
    n_train = int(0.6 * q.shape[0])
    n_calibration = int(0.2 * q.shape[0])
    groups = {
        "train": channel_order[:n_train],
        "calibration": channel_order[n_train:n_train + n_calibration],
        "test": channel_order[n_train + n_calibration:],
    }
    if any(len(group) == 0 for group in groups.values()):
        raise RuntimeError(f"Empty channel partition: {name}")
    if len(set(np.concatenate(list(groups.values())).tolist())) != q.shape[0]:
        raise RuntimeError(f"Channel split is not disjoint: {name}")

    weights_per_channel = q[0].size
    for part, channels in groups.items():
        population = len(channels) * weights_per_channel
        sample_size = min(
            protocol06a["split"]["maximum_target_weights_per_layer"][part],
            population,
        )
        local = rng.choice(population, size=sample_size, replace=False)
        flat = (
            channels[local // weights_per_channel] * weights_per_channel
            + local % weights_per_channel
        )
        integers = q.reshape(-1)[flat].astype(np.int16) & 255
        bits = (
            (integers[:, None] >> np.arange(8)) & 1
        ).astype(np.uint8)
        context = neighbor_features(q, flat)

        # Changing the center must not change its own neighbor features.
        probe_flat = int(flat[0])
        altered = q.copy()
        probe_value = int(altered.reshape(-1)[probe_flat]) & 255
        changed_value = probe_value ^ 128
        altered.reshape(-1)[probe_flat] = (
            changed_value - 256 if changed_value >= 128 else changed_value
        )
        if not np.array_equal(
            context[:1], neighbor_features(altered, flat[:1])
        ):
            raise RuntimeError("Center leaked into neighbor features.")

        values = {
            "bits": bits,
            "context": context,
            "layer": np.full(sample_size, layer_id, dtype=np.int16),
            "channel": (flat // weights_per_channel).astype(np.int32),
            "flat_index": flat.astype(np.int64),
        }
        for key, value in values.items():
            collected[part][key].append(value)

    split_inventory.append({
        "layer_id": layer_id,
        "layer": name,
        "integer_array_sha256": hashlib.sha256(q.tobytes()).hexdigest(),
        "channels": {part: group.tolist() for part, group in groups.items()},
    })

arrays06a = {
    f"{part}_{key}": np.concatenate(values)
    for part, fields in collected.items()
    for key, values in fields.items()
}
buffer = io.BytesIO()
np.savez_compressed(buffer, **arrays06a)
dataset_bytes = buffer.getvalue()

manifest06a = {
    "status": "DATA_PREPARED_NO_PREDICTOR_TRAINED",
    "protocol_sha256": hashlib.sha256(protocol_bytes).hexdigest(),
    "dataset_sha256": hashlib.sha256(dataset_bytes).hexdigest(),
    "numpy_version": np.__version__,
    "sampling": "PCG64; per-layer seed from first 16 hex digits of SHA256([seed, layer])",
    "channel_rounding": "floor_0.6_train_floor_0.2_calibration_remainder_test",
    "context_columns": [
        "mean", "standard_deviation", "mean_absolute", "zero_fraction",
        *[f"bit_{bit}_frequency" for bit in range(8)],
        "neighbor_count",
    ],
    "partition_counts": {
        part: len(arrays06a[f"{part}_bits"]) for part in partitions
    },
    "inventory": split_inventory,
    "feature_rules": [
        "For bit b, exclude column b from target bits before prediction.",
        "Channel and flat_index are audit metadata, never predictor inputs.",
        "Context excludes the target and stays inside its output channel.",
    ],
}
manifest_bytes = (
    json.dumps(manifest06a, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

output_dir.mkdir(parents=True, exist_ok=False)
for filename, content in [
    ("dataset.npz", dataset_bytes),
    ("manifest.json", manifest_bytes),
]:
    path = output_dir / filename
    with path.open("xb") as handle:
        handle.write(content)
    if path.read_bytes() != content:
        raise RuntimeError(f"Write verification failed: {filename}")

print("Saved:", output_dir)
print("Targets by partition:", manifest06a["partition_counts"])
print("Dataset SHA256:", manifest06a["dataset_sha256"])
print("Manifest SHA256:", hashlib.sha256(manifest_bytes).hexdigest())
print("Channel separation and center-exclusion checks: PASS")
print("Bit labels are stored for supervision; the target bit must be masked.")
print("No predictor trained. No COCO images used. No commit or push.")

Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook06a/masked_bit_dataset_v1
Targets by partition: {'train': 32256, 'calibration': 16034, 'test': 16061}
Dataset SHA256: 6e2a4c366321b392f59a330b2cc2810a965897feb9d72e34e98b492666a1df0e
Manifest SHA256: 515632610a2503bce69fd99b21391d37968d217457c54f39d56f73143ea307fe
Channel separation and center-exclusion checks: PASS
Bit labels are stored for supervision; the target bit must be masked.
No predictor trained. No COCO images used. No commit or push.


In [11]:
from google.colab import files
from pathlib import Path
import hashlib
import io
import subprocess
import sys
import zipfile

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Upload exactly one ZIP file.")

content = next(iter(uploaded.values()))
if hashlib.sha256(content).hexdigest() != (
    "556db05495fe28e68c48527ee019a0f2c37813831bc4def2e556a98b0102bba2"
):
    raise RuntimeError("Package SHA256 mismatch.")

expected = {
    "src/contextual06a_runner_v1.py":
        "e63912dd649a0fda19e0170759b23a060b6966f22753f52fad84a4d27c0caeaf",
    "tests/test_contextual06a_runner_v1.py":
        "0e04dc09d36421a84688503befb7617e88bca7abf4b6dd45c6a3d1874565141f",
    "docs/research_reports/06a_runner_notes_v1.md":
        "74f843750e991dbb49ba8e62defb26474d90f5c35a6ab52483ac1339e820f037",
}

with zipfile.ZipFile(io.BytesIO(content)) as archive:
    names = archive.namelist()
    if len(names) != len(expected) or set(names) != set(expected):
        raise RuntimeError("Unexpected package entries.")

    verified = {}
    for name, expected_sha in expected.items():
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != expected_sha:
            raise RuntimeError(f"File checksum mismatch: {name}")
        destination = repo / name
        if destination.exists() and destination.read_bytes() != data:
            raise RuntimeError(f"Existing file differs: {name}")
        verified[name] = data

    for name, data in verified.items():
        destination = repo / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            with destination.open("xb") as handle:
                handle.write(data)
        if destination.read_bytes() != data:
            raise RuntimeError(f"Installation verification failed: {name}")
        print("Verified:", name)

result = subprocess.run(
    [
        sys.executable, "-m", "unittest", "discover",
        "-s", "tests", "-p", "test_contextual06a_runner_v1.py", "-v",
    ],
    cwd=repo,
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError("Feasibility control tests failed.")

print("Package installed. Synthetic control tests passed.")
print("Synthetic tests include fitting on artificial data only.")
print("Research dataset not evaluated. No COCO images used.")
print("Existing screening code unchanged. No commit or push.")

Saving contextual06a_runner_v1.zip to contextual06a_runner_v1.zip
Verified: src/contextual06a_runner_v1.py
Verified: tests/test_contextual06a_runner_v1.py
Verified: docs/research_reports/06a_runner_notes_v1.md

test_addresses_and_channel_identity_are_not_features (test_contextual06a_runner_v1.FeasibilityControls.test_addresses_and_channel_identity_are_not_features) ... ok
test_bootstrap_keeps_cluster_dependence (test_contextual06a_runner_v1.FeasibilityControls.test_bootstrap_keeps_cluster_dependence) ... ok
test_constant_label_group_has_no_fake_balanced_accuracy (test_contextual06a_runner_v1.FeasibilityControls.test_constant_label_group_has_no_fake_balanced_accuracy) ... ok
test_context_is_only_extra_feature_block (test_contextual06a_runner_v1.FeasibilityControls.test_context_is_only_extra_feature_block) ... ok
test_exclusive_records_and_checksum_detection (test_contextual06a_runner_v1.FeasibilityControls.test_exclusive_records_and_checksum_detection) ... ok
test_frequency_predictions_

In [12]:
from pathlib import Path
import hashlib
import importlib.util

root = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
module_path = root / "code/src/contextual06a_runner_v1.py"

if hashlib.sha256(module_path.read_bytes()).hexdigest() != (
    "e63912dd649a0fda19e0170759b23a060b6966f22753f52fad84a4d27c0caeaf"
):
    raise RuntimeError("Feasibility runner SHA256 mismatch.")

spec = importlib.util.spec_from_file_location(
    "contextual06a_runner_v1", module_path
)
runner06a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner06a)

result06a = runner06a.run(root)

print("\n=== Masked-bit feasibility completed ===")
print("Status:", result06a["status"])

for name, comparison in (
    result06a["paired_test_log_loss_comparisons"].items()
):
    print(f"\n{name}")
    print("Mean log-loss difference:", comparison["mean_over_bits"])
    print(
        "Exploratory 95% interval:",
        comparison[
            "exploratory_percentile_95_interval_mean_over_bits"
        ],
    )

print("\nNegative differences favor method C.")
print("This measures masked-bit prediction, not fault-repair success.")
print("No COCO images evaluated. No commit or push performed.")

Bit 0: A/B/C/D saved | fit=4.79s
Bit 1: A/B/C/D saved | fit=3.37s
Bit 2: A/B/C/D saved | fit=3.47s
Bit 3: A/B/C/D saved | fit=4.49s
Bit 4: A/B/C/D saved | fit=3.77s
Bit 5: A/B/C/D saved | fit=4.47s
Bit 6: A/B/C/D saved | fit=4.44s
Bit 7: A/B/C/D saved | fit=4.61s
Saved report: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/docs/research_reports/notebook06a_feasibility_v1/feasibility_report_v1.md
No COCO images, network inference, fault repair, commit or push.

=== Masked-bit feasibility completed ===
Status: MASKED_BIT_FEASIBILITY_COMPLETE_NOT_FAULT_REPAIR

C_minus_B
Mean log-loss difference: -0.06258614794455145
Exploratory 95% interval: [-0.06555683940775589, -0.059643395368806265]

C_minus_D
Mean log-loss difference: -0.0099558413454231
Exploratory 95% interval: [-0.011044272597539833, -0.008895462085323819]

Negative differences favor method C.
This measures masked-bit prediction, not fault-repair success.
No COCO images evaluated. No commit or push performed.


In [13]:
from pathlib import Path
import hashlib
import json

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = (
    repo / "docs/research_reports/notebook06a_feasibility_v1/"
    "comparison_summary_v1.json"
)
content = path.read_bytes()
summary = json.loads(content)

expected_sources = {
    "source_sha256":
        "e63912dd649a0fda19e0170759b23a060b6966f22753f52fad84a4d27c0caeaf",
    "protocol_sha256":
        "d851b7a91569ffa155d3e85ffd7afa1fb80e0df851857979cde7a51e392bceec",
    "dataset_sha256":
        "6e2a4c366321b392f59a330b2cc2810a965897feb9d72e34e98b492666a1df0e",
}

for key, expected in expected_sources.items():
    if summary["session"][key] != expected:
        raise RuntimeError(f"Report source mismatch: {key}")

intervals = summary["paired_test_log_loss_comparisons"]["C_minus_D"][
    "exploratory_percentile_95_interval_by_bit"
]

print("Bit | A loss | B loss | D loss | C loss | C-D | Exploratory interval")
for record in summary["bits"]:
    bit = record["bit"]
    losses = {
        method: record["metrics"][f"test_{method}"]["log_loss"]
        for method in ("A", "B", "C", "D")
    }
    print(
        f"{bit} | {losses['A']:.6f} | {losses['B']:.6f} | "
        f"{losses['D']:.6f} | {losses['C']:.6f} | "
        f"{losses['C'] - losses['D']:+.6f} | {intervals[bit]}"
    )

print(
    "\nSaved predictors total storage (MiB):",
    round(summary["models_total_storage_bytes"] / 2**20, 3),
)
print("Report SHA256:", hashlib.sha256(content).hexdigest())
print("Intervals are exploratory and not adjusted for multiple comparisons.")
print("No retraining, fault repair, file changes, commit or push.")

Bit | A loss | B loss | D loss | C loss | C-D | Exploratory interval
0 | 0.694121 | 0.725799 | 0.692734 | 0.694209 | +0.001474 | [0.00019862205237791006, 0.0028579524125195085]
1 | 0.694364 | 0.725124 | 0.690394 | 0.690737 | +0.000343 | [-0.0009643882722833248, 0.00169277210242549]
2 | 0.693424 | 0.710093 | 0.676118 | 0.671789 | -0.004329 | [-0.005988917374551234, -0.002628369353430361]
3 | 0.693239 | 0.672578 | 0.641370 | 0.631008 | -0.010362 | [-0.013365111292567396, -0.0075953258843445845]
4 | 0.693682 | 0.590449 | 0.557289 | 0.535103 | -0.022186 | [-0.025551711116268336, -0.018770405050213716]
5 | 0.694044 | 0.434984 | 0.392365 | 0.365096 | -0.027269 | [-0.03119410466579659, -0.023395611507460903]
6 | 0.692755 | 0.255471 | 0.154005 | 0.141798 | -0.012207 | [-0.015566576045135502, -0.009323495171277937]
7 | 0.693250 | 0.226168 | 0.115348 | 0.110238 | -0.005110 | [-0.0075622827135347175, -0.0027730099879402386]

Saved predictors total storage (MiB): 1.171
Report SHA256: 95a748398cda0

In [14]:
from pathlib import Path
import hashlib
import json

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
source = (
    repo / "docs/research_reports/notebook06a_feasibility_v1/"
    "comparison_summary_v1.json"
)
source_bytes = source.read_bytes()
source_sha = hashlib.sha256(source_bytes).hexdigest()

if source_sha != (
    "95a748398cda073cb13353760647dd82a9e4de71c2fea02daacbbe07c5c851ba"
):
    raise RuntimeError("Comparison summary SHA256 mismatch.")

summary = json.loads(source_bytes)
comparison = summary["paired_test_log_loss_comparisons"]["C_minus_D"]

lines = [
    "# Contextual masked-bit feasibility: result card",
    "",
    "Status: initial feasibility completed; not fault detection or repair.",
    f"Source report SHA256: {source_sha}",
    "",
    "C and D use the same learner and settings. C adds local context.",
    "Output channels are disjoint between training, calibration and test.",
    "",
    "| Bit | D test log loss | C test log loss | C minus D |",
    "| ---: | ---: | ---: | ---: |",
]

for record in summary["bits"]:
    metrics = record["metrics"]
    d = metrics["test_D"]["log_loss"]
    c = metrics["test_C"]["log_loss"]
    lines.append(f"| {record['bit']} | {d:.9f} | {c:.9f} | {c-d:+.9f} |")

lines += [
    "",
    f"Mean C-minus-D log-loss difference: {comparison['mean_over_bits']}",
    "Exploratory interval: "
    + str(comparison["exploratory_percentile_95_interval_mean_over_bits"]),
    "",
    "## Interpretation",
    "",
    "- Context reduced observed test log loss for bits 2 through 7.",
    "- Bit 0 worsened; bit 1's exploratory interval includes zero.",
    "- Retain all bit results; do not select successful bits and report them as a preregistered subset.",
    "- Intervals are conditional, exploratory and unadjusted for multiple comparisons.",
    "- No unknown-fault detection, correction, false-alarm or AP-recovery result exists yet.",
    "- Clean context and a known masked-bit position were assumed.",
    "- Compressed predictor storage is not total deployment overhead.",
    "- The test set must not be reused for tuning this comparison.",
    "",
    "Next: checkpoint this evidence, then specify a separate fault-detection",
    "and repair feasibility protocol with independent faults and no-fault controls.",
    "",
]

content = "\n".join(lines).encode("utf-8")
destination = (
    repo / "docs/research_reports/06a_contextual_bit_feasibility_card_v1.md"
)

if destination.exists():
    if destination.read_bytes() != content:
        raise RuntimeError("Existing result card differs; do not overwrite.")
else:
    with destination.open("xb") as handle:
        handle.write(content)

if destination.read_bytes() != content:
    raise RuntimeError("Result card verification failed.")

print("Saved:", destination)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No retraining or repair. No commit or push performed.")

Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/docs/research_reports/06a_contextual_bit_feasibility_card_v1.md
SHA256: 042119f788ec99759c9168a544176216263c77fa23a45a4d87985310dfa60ddb
No retraining or repair. No commit or push performed.
